# 01_train_template

Duplicate this notebook per experiment, rename to the target run name, edit ONLY the `RunConfig` cell, then Run All.

Artifacts produced:
- `models/<cfg.name>/` — trained weights + tokenizer
- `results/<cfg.name>.json` — per-run classification reports + saved config
- `results/<cfg.name>.xlsx` — per-run summary + per-class details
- `results/models_db.json` — central BERT-side DB entry
- `results/master_summary.xlsx` — rebuilt aggregate over all runs (BERT + Yandex)


In [ ]:
# Bootstrap: make the project root importable no matter where this notebook lives.
import os, sys
_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'pipeline')) else os.path.abspath(os.path.join(_here, os.pardir))
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)
print('project root:', _root)

## Run config — **the only cell you edit per experiment**

In [ ]:
from pipeline.env import DATA_DIR, MODEL_DIR
from pipeline.config import RunConfig

cfg = RunConfig(
    name          = "01_baseline_focal",                    # CHANGE-ME per run
    architecture  = "bert",                                 # "bert" | "bert+crf" | "bert+lstm"
    loss          = "focal",                                # "ce" | "focal" | "crf"
    train_files   = [f"{DATA_DIR}/train_all.json"],
    val_files     = [f"{DATA_DIR}/val_internal.json"],
    replay_files  = [],                                     # e.g. [{"path": f"{DATA_DIR}/train_all_plus_synth.json", "n": 15000, "seed": 42}]
    init_from     = None,                                   # e.g. f"{MODEL_DIR}/02_synth_focal" to warm-start
    num_epochs    = 3,
    learning_rate = 2e-5,
    tags          = {"stage": "1-baseline-loss-sweep"},

    # --- Advanced knobs (leave off for clean baselines; see RunConfig for details)
    # crf_init_transitions = True,      # warm-start CRF transition matrix from train data (bert+crf only)
    # crf_aux_loss         = "focal",   # "none" | "ce_weighted" | "focal" — per-token aux loss on CRF emissions
    # crf_aux_weight       = 0.2,       # lambda on the aux term
    # crf_aux_gamma        = 2.0,       # focal gamma (ignored if aux="ce_weighted")
    # o_mask_prob          = 0.3,       # p of O tokens masked per batch (CE/focal only; no-op for bert+crf)
)
print(cfg)

## Train

In [ ]:
from pipeline.training import train_run
model = train_run(cfg)

## Benchmark + save results

Scores on `bench_test_all`, `bench_test_gera`, `bench_test_lorugec`. Writes per-run json/xlsx + updates `results/models_db.json`.

In [ ]:
from pipeline.evaluation import evaluate_and_save
reports = evaluate_and_save(cfg)

for test_name, rep in reports.items():
    wa = rep.get('weighted avg', {})
    print(f"{test_name:>14s} | F1={wa.get('f1-score', 0):.4f} P={wa.get('precision', 0):.4f} R={wa.get('recall', 0):.4f}")

## Qualitative demo (optional)

In [ ]:
from pipeline.inference import load_for_inference, restore_punctuation

m, tok = load_for_inference(cfg)
for t in [
    "Мама мыла раму а папа читал газету",
    "Однако мы решили не идти в кино потому что пошел дождь",
    "Он сказал Привет как дела",
    "Я говорю ей Что думаешь А она мне Да ничего отстань уже",
    "После многих страданий А А Пушкин всё-таки написал свои стишки",
]:
    print(f"Input:    {t}")
    print(f"Restored: {restore_punctuation(m, tok, t)}\n")

## Rebuild master summary across ALL runs (BERT + Yandex)

In [ ]:
from pipeline.aggregate import rebuild_master_excel
rebuild_master_excel()